# FrontierMem Seed 17 — Stage A: CAID training ONLY (Pointwise never retrained)

### Purpose

**ONLY** train Qwen3-4B CAID Seed 17. The existing Pointwise Seed 17 is not loaded, evaluated, or retrained here. After training, download the CAID checkpoint ZIP **before** starting any evaluation. This protects the expensive GPU work from Kaggle session limits.

GPU + Internet ON; if the repo is private, Kaggle Secret `GITHUB_TOKEN`.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json
REPO_URL='https://github.com/MohammadAmanlou/FrontierMem.git'
REPO_COMMIT='fca62bf316fabd629edf78ea46a36620d331ffe2'
MODEL='Qwen/Qwen3-4B-Instruct-2507'
SEED=17
WORKING=Path('/kaggle/working'); PROJECT=WORKING/'FrontierMem'
RESULTS=WORKING/'frontiermem_seed17_no_repeat'

def run(cmd,cwd=None,check=True,env=None):
    print('\n$ '+str(cmd)+'\n',flush=True)
    return subprocess.run(cmd,shell=True,cwd=cwd,env=env,check=check)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass
if token:
    askpass=WORKING/'.fm_askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token";;\n*Password*) echo "$GITHUB_TOKEN";;\n*) echo "";;\nesac\n')
    askpass.chmod(0o700)
    env=os.environ.copy();env['GITHUB_TOKEN']=token
    env['GIT_ASKPASS']=str(askpass);env['GIT_TERMINAL_PROMPT']='0'
    try: run(f'git clone {REPO_URL} {PROJECT}',env=env)
    finally: askpass.unlink(missing_ok=True)
else:
    run(f'git clone {REPO_URL} {PROJECT}')
run(f'git checkout {REPO_COMMIT}',cwd=PROJECT)
run(f'{sys.executable} -m pip uninstall -y torchao',check=False)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.57,<5" "peft>=0.20,<0.21" "bitsandbytes>=0.46" "accelerate>=1.0" "pandas>=2.2,<2.4" "scikit-learn>=1.4,<1.9"')
run(f'{sys.executable} -m pip install -q -r requirements-train.txt',cwd=PROJECT)
run(f'{sys.executable} -m pip install -q -e .',cwd=PROJECT)
import torch
assert torch.cuda.is_available(),'Enable GPU in Kaggle settings.'
print('GPU:',torch.cuda.get_device_name(0))
RESULTS.mkdir(parents=True,exist_ok=True)
summary=json.loads((PROJECT/'data/external_processed/summary.json').read_text())
assert summary['n_atomic_examples']==12423
assert summary['by_source']['benchpres']==1950
assert sum(1 for ln in open(PROJECT/'data/external_processed/caid_pairwise_train.jsonl',encoding='utf-8') if ln.strip())==6336
print('Dataset assertions passed; pinned commit:', REPO_COMMIT)

### Install the existing strong CAID trainer locally (nothing is pushed to GitHub).

In [ ]:
SCRIPT = '#!/usr/bin/env python\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom torch.utils.data import DataLoader, Dataset\n\nLABEL2ID = {"IGNORE": 0, "SUPPORT": 1, "DOMINATE": 2}\nID2LABEL = {v: k for k, v in LABEL2ID.items()}\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef load_jsonl(path):\n    rows = []\n    with Path(path).open("r", encoding="utf-8") as f:\n        for line in f:\n            if line.strip():\n                rows.append(json.loads(line))\n    return rows\n\n\ndef format_side(side: dict) -> str:\n    parts = [f"[PREFERENCE]\\n{side.get(\'preference\', \'\')}"]\n    history = str(side.get("history", ""))\n    context = str(side.get("context", ""))\n    if history.strip():\n        parts.append(f"[HISTORY]\\n{history}")\n    if context.strip():\n        parts.append(f"[CONTEXT]\\n{context}")\n    parts.append(f"[QUERY]\\n{side.get(\'query\', \'\')}")\n    parts.append(\n        "[TASK]\\nJudge the preference-query relation using the native RPEval "\n        "labels IGNORE, SUPPORT, or DOMINATE."\n    )\n    parts.append("[DECISION]")\n    return "\\n".join(parts)\n\n\ndef group_safe_pair_split(pairs, dev_fraction: float, seed: int):\n    groups = sorted({str(p.get("group_id", "")) for p in pairs})\n    if not groups or "" in groups:\n        raise SystemExit("Pair file is missing group_id.")\n    rng = random.Random(seed)\n    rng.shuffle(groups)\n    dev_n = max(1, int(round(dev_fraction * len(groups))))\n    dev_groups = set(groups[:dev_n])\n    train = [p for p in pairs if str(p["group_id"]) not in dev_groups]\n    dev = [p for p in pairs if str(p["group_id"]) in dev_groups]\n    if {p["group_id"] for p in train} & {p["group_id"] for p in dev}:\n        raise RuntimeError("Group leakage detected.")\n    return train, dev\n\n\nclass FlatPairDataset(Dataset):\n    def __init__(self, pairs):\n        self.items = []\n        for pair in pairs:\n            for name in ("positive", "negative"):\n                side = pair[name]\n                label = str(side.get("native_label", "")).upper()\n                if label in LABEL2ID:\n                    self.items.append((format_side(side), LABEL2ID[label]))\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        return self.items[i]\n\n\nclass PairDataset(Dataset):\n    def __init__(self, rows):\n        self.rows = rows\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, i):\n        row = self.rows[i]\n        return {\n            "positive_text": format_side(row["positive"]),\n            "positive_label": LABEL2ID[str(row["positive"]["native_label"]).upper()],\n            "negative_text": format_side(row["negative"]),\n            "negative_label": LABEL2ID[str(row["negative"]["native_label"]).upper()],\n        }\n\n\nclass FlatCollator:\n    def __init__(self, tokenizer, max_length):\n        self.tok = tokenizer\n        self.max_length = max_length\n\n    def __call__(self, batch):\n        texts, labels = zip(*batch)\n        toks = self.tok(\n            list(texts), padding=True, truncation=True,\n            max_length=self.max_length, return_tensors="pt"\n        )\n        return toks, torch.tensor(labels, dtype=torch.long)\n\n\nclass PairConcatCollator:\n    """Tokenize positive and negative sides together so CAID uses one forward call."""\n    def __init__(self, tokenizer, max_length):\n        self.tok = tokenizer\n        self.max_length = max_length\n\n    def __call__(self, batch):\n        pos_text = [x["positive_text"] for x in batch]\n        neg_text = [x["negative_text"] for x in batch]\n        texts = pos_text + neg_text\n        labels = (\n            [x["positive_label"] for x in batch]\n            + [x["negative_label"] for x in batch]\n        )\n        toks = self.tok(\n            texts, padding=True, truncation=True,\n            max_length=self.max_length, return_tensors="pt"\n        )\n        return toks, torch.tensor(labels, dtype=torch.long), len(batch)\n\n\ndef applicability_score(logits):\n    z = logits.float()\n    return torch.logsumexp(z[:, 1:3], dim=-1) - z[:, 0]\n\n\ndef binary_anchor_loss(logits, labels):\n    y = (labels != LABEL2ID["IGNORE"]).float()\n    return F.binary_cross_entropy_with_logits(applicability_score(logits), y)\n\n\ndef native_loss(logits, labels, smoothing):\n    return F.cross_entropy(logits.float(), labels, label_smoothing=smoothing)\n\n\ndef build_model(args, tokenizer):\n    from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig\n    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n\n    compute_dtype = torch.float16  # T4\n    qcfg = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=compute_dtype,\n    )\n    model = AutoModelForSequenceClassification.from_pretrained(\n        args.model,\n        num_labels=3,\n        id2label=ID2LABEL,\n        label2id=LABEL2ID,\n        quantization_config=qcfg,\n        device_map={"": 0},\n        dtype=compute_dtype,\n        trust_remote_code=True,\n        attn_implementation="sdpa",\n    )\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n\n    model = prepare_model_for_kbit_training(\n        model,\n        use_gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n\n    lora = LoraConfig(\n        task_type=TaskType.SEQ_CLS,\n        r=args.lora_r,\n        lora_alpha=args.lora_alpha,\n        lora_dropout=args.lora_dropout,\n        target_modules="all-linear",\n        modules_to_save=["score"],\n        bias="none",\n        use_rslora=True,\n    )\n    model = get_peft_model(model, lora)\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    return model, compute_dtype\n\n\ndef build_optimizer(model, args):\n    import bitsandbytes as bnb\n    groups = {"A": [], "B": [], "head": [], "other": []}\n    for name, p in model.named_parameters():\n        if not p.requires_grad:\n            continue\n        low = name.lower()\n        if "lora_a" in low:\n            groups["A"].append(p)\n        elif "lora_b" in low:\n            groups["B"].append(p)\n        elif "score" in low:\n            groups["head"].append(p)\n        else:\n            groups["other"].append(p)\n\n    pgs = []\n    if groups["A"]:\n        pgs.append({"params": groups["A"], "lr": args.lr_a, "weight_decay": args.weight_decay})\n    if groups["B"]:\n        pgs.append({"params": groups["B"], "lr": args.lr_a * args.loraplus_ratio, "weight_decay": args.weight_decay})\n    if groups["head"]:\n        pgs.append({"params": groups["head"], "lr": args.head_lr, "weight_decay": args.weight_decay})\n    if groups["other"]:\n        pgs.append({"params": groups["other"], "lr": args.lr_a, "weight_decay": args.weight_decay})\n    return bnb.optim.PagedAdamW8bit(pgs, betas=(0.9, 0.999), eps=1e-8)\n\n\n@torch.no_grad()\ndef evaluate_dev(model, pair_loader, device):\n    model.eval()\n    nt, npred, bt, bpred, margins = [], [], [], [], []\n    for toks, labels, n_pairs in pair_loader:\n        toks = {k: v.to(device, non_blocking=True) for k, v in toks.items()}\n        labels = labels.to(device, non_blocking=True)\n        logits = model(**toks).logits.float()\n        lp, ln = logits[:n_pairs], logits[n_pairs:]\n        yp, yn = labels[:n_pairs], labels[n_pairs:]\n\n        nt.extend(yp.cpu().tolist()); nt.extend(yn.cpu().tolist())\n        npred.extend(lp.argmax(-1).cpu().tolist()); npred.extend(ln.argmax(-1).cpu().tolist())\n\n        sp, sn = applicability_score(lp), applicability_score(ln)\n        bt.extend([1] * n_pairs + [0] * n_pairs)\n        bpred.extend((sp >= 0).long().cpu().tolist() + (sn >= 0).long().cpu().tolist())\n        margins.extend((sp - sn).cpu().tolist())\n\n    return {\n        "dev_native_3way_accuracy": float(accuracy_score(nt, npred)),\n        "dev_native_3way_macro_f1": float(f1_score(nt, npred, labels=[0,1,2], average="macro", zero_division=0)),\n        "dev_binary_accuracy": float(accuracy_score(bt, bpred)),\n        "dev_binary_macro_f1": float(f1_score(bt, bpred, labels=[1,0], average="macro", zero_division=0)),\n        "dev_pair_direction_accuracy": float(np.mean(np.asarray(margins) > 0)),\n        "dev_mean_applicability_margin": float(np.mean(margins)),\n        "dev_median_applicability_margin": float(np.median(margins)),\n    }\n\n\ndef save_best(model, tokenizer, best_dir, record, args):\n    if best_dir.exists():\n        shutil.rmtree(best_dir)\n    best_dir.mkdir(parents=True, exist_ok=True)\n    model.save_pretrained(best_dir)\n    tokenizer.save_pretrained(best_dir)\n    (best_dir / "best_meta.json").write_text(json.dumps(record, indent=2), encoding="utf-8")\n    (best_dir / "training_config.json").write_text(json.dumps(vars(args), indent=2), encoding="utf-8")\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--method", choices=["pointwise", "caid"], required=True)\n    p.add_argument("--pairs", default="data/external_processed/caid_pairwise_train.jsonl")\n    p.add_argument("--model", default="Qwen/Qwen3-4B-Instruct-2507")\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--max-epochs", type=int, default=3)\n    p.add_argument("--patience", type=int, default=1)\n    p.add_argument("--pointwise-batch", type=int, default=4)\n    p.add_argument("--caid-pair-batch", type=int, default=2)\n    p.add_argument("--grad-accum", type=int, default=4)\n    p.add_argument("--lr-a", type=float, default=1e-5)\n    p.add_argument("--loraplus-ratio", type=float, default=16.0)\n    p.add_argument("--head-lr", type=float, default=2e-4)\n    p.add_argument("--weight-decay", type=float, default=0.01)\n    p.add_argument("--warmup-ratio", type=float, default=0.05)\n    p.add_argument("--max-length", type=int, default=1024)\n    p.add_argument("--label-smoothing", type=float, default=0.05)\n    p.add_argument("--lambda-binary", type=float, default=0.5)\n    p.add_argument("--lambda-rank", type=float, default=0.5)\n    p.add_argument("--rank-margin", type=float, default=0.5)\n    p.add_argument("--rank-temperature", type=float, default=1.0)\n    p.add_argument("--lora-r", type=int, default=32)\n    p.add_argument("--lora-alpha", type=int, default=64)\n    p.add_argument("--lora-dropout", type=float, default=0.05)\n    p.add_argument("--seed", type=int, default=17)\n    p.add_argument("--dev-fraction", type=float, default=0.10)\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--progress-every", type=int, default=100)\n    args = p.parse_args()\n\n    seed_all(args.seed)\n    from transformers import AutoTokenizer, get_cosine_schedule_with_warmup\n\n    rows = load_jsonl(args.pairs)\n    train_rows, dev_rows = group_safe_pair_split(rows, args.dev_fraction, args.seed)\n    print(json.dumps({\n        "seed": args.seed, "method": args.method,\n        "train_pairs": len(train_rows), "dev_pairs": len(dev_rows),\n        "train_groups": len({p["group_id"] for p in train_rows}),\n        "dev_groups": len({p["group_id"] for p in dev_rows}),\n        "group_overlap": 0,\n    }, indent=2), flush=True)\n\n    tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n\n    model, compute_dtype = build_model(args, tok)\n    model.print_trainable_parameters()\n    device = torch.device("cuda:0")\n\n    pair_collator = PairConcatCollator(tok, args.max_length)\n    dev_loader = DataLoader(\n        PairDataset(dev_rows),\n        batch_size=args.caid_pair_batch,\n        shuffle=False,\n        collate_fn=pair_collator,\n        pin_memory=True,\n        num_workers=args.num_workers,\n        persistent_workers=args.num_workers > 0,\n    )\n\n    if args.method == "pointwise":\n        train_loader = DataLoader(\n            FlatPairDataset(train_rows),\n            batch_size=args.pointwise_batch,\n            shuffle=True,\n            collate_fn=FlatCollator(tok, args.max_length),\n            pin_memory=True,\n            num_workers=args.num_workers,\n            persistent_workers=args.num_workers > 0,\n        )\n        sequences_per_microbatch = args.pointwise_batch\n    else:\n        train_loader = DataLoader(\n            PairDataset(train_rows),\n            batch_size=args.caid_pair_batch,\n            shuffle=True,\n            collate_fn=pair_collator,\n            pin_memory=True,\n            num_workers=args.num_workers,\n            persistent_workers=args.num_workers > 0,\n        )\n        sequences_per_microbatch = 2 * args.caid_pair_batch\n\n    opt = build_optimizer(model, args)\n    updates_per_epoch = math.ceil(len(train_loader) / args.grad_accum)\n    total_updates = updates_per_epoch * args.max_epochs\n    warmup = max(1, int(round(args.warmup_ratio * total_updates)))\n    sched = get_cosine_schedule_with_warmup(opt, warmup, total_updates)\n\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    best_dir = out / "best"\n\n    (out / "run_manifest.json").write_text(json.dumps({\n        "method": args.method,\n        "seed": args.seed,\n        "model": args.model,\n        "single_forward_for_caid_pos_neg": True,\n        "sequences_per_microbatch": sequences_per_microbatch,\n        "effective_sequences_per_update": sequences_per_microbatch * args.grad_accum,\n        "gradient_checkpointing_use_reentrant": False,\n        "selection_metric": "dev_binary_macro_f1",\n        "tie_breakers": ["dev_native_3way_macro_f1", "dev_pair_direction_accuracy"],\n        "max_epochs": args.max_epochs,\n        "patience": args.patience,\n    }, indent=2), encoding="utf-8")\n\n    history = []\n    best_tuple = None\n    best_epoch = None\n    bad = 0\n    global_update = 0\n\n    for epoch in range(1, args.max_epochs + 1):\n        model.train()\n        opt.zero_grad(set_to_none=True)\n        start = time.time()\n        running = {"loss":0.0, "native":0.0, "binary":0.0, "rank":0.0}\n        nb = 0\n\n        for step, batch in enumerate(train_loader, start=1):\n            if args.method == "pointwise":\n                toks, labels = batch\n                toks = {k:v.to(device, non_blocking=True) for k,v in toks.items()}\n                labels = labels.to(device, non_blocking=True)\n                with torch.autocast("cuda", dtype=compute_dtype):\n                    logits = model(**toks).logits\n                nloss = native_loss(logits, labels, args.label_smoothing)\n                bloss = binary_anchor_loss(logits, labels)\n                rloss = torch.zeros((), device=device)\n                loss = nloss + args.lambda_binary * bloss\n            else:\n                toks, labels, n_pairs = batch\n                toks = {k:v.to(device, non_blocking=True) for k,v in toks.items()}\n                labels = labels.to(device, non_blocking=True)\n                with torch.autocast("cuda", dtype=compute_dtype):\n                    logits = model(**toks).logits\n                lp, ln = logits[:n_pairs], logits[n_pairs:]\n                yp, yn = labels[:n_pairs], labels[n_pairs:]\n                nloss = (native_loss(lp, yp, args.label_smoothing) + native_loss(ln, yn, args.label_smoothing))/2\n                bloss = (binary_anchor_loss(lp, yp) + binary_anchor_loss(ln, yn))/2\n                delta = applicability_score(lp) - applicability_score(ln)\n                rloss = F.softplus((args.rank_margin-delta)/args.rank_temperature).mean()*args.rank_temperature\n                loss = nloss + args.lambda_binary*bloss + args.lambda_rank*rloss\n\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Non-finite loss at epoch={epoch}, step={step}")\n\n            (loss / args.grad_accum).backward()\n\n            if step % args.grad_accum == 0 or step == len(train_loader):\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                opt.step()\n                sched.step()\n                opt.zero_grad(set_to_none=True)\n                global_update += 1\n\n            running["loss"] += float(loss.detach().cpu())\n            running["native"] += float(nloss.detach().cpu())\n            running["binary"] += float(bloss.detach().cpu())\n            running["rank"] += float(rloss.detach().cpu())\n            nb += 1\n\n            if step % args.progress_every == 0 or step == len(train_loader):\n                elapsed = time.time() - start\n                rate = elapsed / step\n                eta = rate * (len(train_loader) - step)\n                print(json.dumps({\n                    "epoch": epoch,\n                    "step": step,\n                    "steps": len(train_loader),\n                    "pct": round(100*step/len(train_loader), 1),\n                    "elapsed_min": round(elapsed/60, 1),\n                    "eta_min": round(eta/60, 1),\n                    "avg_loss_so_far": round(running["loss"]/nb, 5),\n                }), flush=True)\n\n        dev = evaluate_dev(model, dev_loader, device)\n        rec = {\n            "epoch": epoch,\n            "global_updates": global_update,\n            "epoch_minutes": round((time.time()-start)/60, 2),\n            "train_loss": running["loss"]/max(1,nb),\n            "train_native_loss": running["native"]/max(1,nb),\n            "train_binary_anchor_loss": running["binary"]/max(1,nb),\n            "train_rank_loss": running["rank"]/max(1,nb),\n            **dev,\n        }\n        history.append(rec)\n        print(json.dumps(rec, indent=2), flush=True)\n\n        candidate = (\n            rec["dev_binary_macro_f1"],\n            rec["dev_native_3way_macro_f1"],\n            rec["dev_pair_direction_accuracy"],\n        )\n        if best_tuple is None or candidate > best_tuple:\n            best_tuple = candidate\n            best_epoch = epoch\n            bad = 0\n            save_best(model, tok, best_dir, rec, args)\n            print(f"NEW BEST epoch={epoch}: {best_tuple}", flush=True)\n        else:\n            bad += 1\n            print(f"No improvement: patience {bad}/{args.patience}", flush=True)\n\n        (out / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n\n        if bad >= args.patience:\n            print(f"EARLY STOP. Best epoch={best_epoch}.", flush=True)\n            break\n\n    summary = {\n        "method": args.method,\n        "seed": args.seed,\n        "best_epoch": best_epoch,\n        "best_selection_tuple": best_tuple,\n        "epochs_ran": len(history),\n        "best_checkpoint": str(best_dir),\n    }\n    (out / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2), flush=True)\n\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nif __name__ == "__main__":\n    main()\n'
p=(PROJECT/'scripts'/'v4_1_train_fast_strong.py')
p.write_text(SCRIPT,encoding='utf-8')
compile(SCRIPT,str(p),'exec')
print('Script installed:',p)

### Run CAID training. Best dev checkpoint is saved at the end of every improving epoch.

In [ ]:
CAID_TRAIN=RESULTS/'caid'/'train'
if (CAID_TRAIN/'training_summary.json').exists():
    print('CAID already trained in this session: SKIP')
elif (CAID_TRAIN/'best'/'adapter_model.safetensors').exists():
    raise RuntimeError('A partial CAID best checkpoint exists. Preserve it instead of silently repeating the run.')
else:
    run(
        f'{sys.executable} scripts/v4_1_train_fast_strong.py '
        f'--method caid --output-dir {CAID_TRAIN} '
        '--pairs data/external_processed/caid_pairwise_train.jsonl '
        f'--model {MODEL} '
        '--max-epochs 5 --patience 2 '
        '--pointwise-batch 4 --caid-pair-batch 2 --grad-accum 4 '
        '--max-length 1024 --lr-a 1e-5 --loraplus-ratio 16 '
        '--head-lr 2e-4 --weight-decay 0.01 --warmup-ratio 0.05 '
        '--label-smoothing 0.05 --lambda-binary 0.5 '
        '--lambda-rank 0.5 --rank-margin 0.5 --rank-temperature 1.0 '
        '--lora-r 32 --lora-alpha 64 --lora-dropout 0.05 '
        f'--seed {SEED} --dev-fraction 0.10 --num-workers 2 --progress-every 100',
        cwd=PROJECT,
    )
assert (CAID_TRAIN/'best'/'adapter_model.safetensors').is_file()
assert (CAID_TRAIN/'training_summary.json').is_file()
print('CAID training finished. Best epoch:',json.loads((CAID_TRAIN/'training_summary.json').read_text())['best_epoch'])

### Save the CAID checkpoint and training logs before evaluation.

In [ ]:
import zipfile,hashlib
CAID_ZIP=WORKING/'FrontierMem_Caid_Seed17_Complete_Training_Checkpoint.zip'
with zipfile.ZipFile(CAID_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in CAID_TRAIN.rglob('*'):
        if p.is_file():
            z.write(p,p.relative_to(RESULTS.parent))
with zipfile.ZipFile(CAID_ZIP) as z:
    assert z.testzip() is None
    assert any(n.endswith('adapter_model.safetensors') for n in z.namelist())
print('DOWNLOAD THIS FIRST:',CAID_ZIP)
print('MB:',round(CAID_ZIP.stat().st_size/1e6,1))

**Stop here.** Download the CAID checkpoint ZIP from Kaggle Output and keep it with the Pointwise checkpoint ZIP. Next, run **Stage B**. If Stage A does not complete, do not treat a partial best checkpoint as a completed training run.